# Buscador de parámetros de servicio

Este cuaderno llama `service_parameter_search.py` para ajustar parámetros de un servicio bajo restricciones de bote, carril y altura. La barra inferior muestra el avance de cada generación y del pulido local. Si SciPy no está instalado, utiliza la búsqueda aleatoria incluida y muestra el avance por muestras evaluadas.


In [ ]:
import json
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

import benchmark_direct_services as benchmark_direct
from service_parameter_search import (
    RacketSearchConfig,
    SearchConfig,
    ServeTargets,
    SearchWeights,
    benchmark_initial_guess,
    benchmark_racket_initial_guess,
    search_direct_parameters,
    search_racket_parameters,
    target_from_benchmark,
)
from table_tennis_simulation import TABLE_LENGTH, TABLE_WIDTH


In [ ]:
services = list(benchmark_direct.SERVICE_TYPES.keys())
depths = list(benchmark_direct.DEPTHS.keys())
lanes = list(benchmark_direct.LANES.keys())

mode = widgets.ToggleButtons(options=[('Directo', 'direct'), ('Raqueta', 'racket')], value='direct', description='Modo')
service = widgets.Dropdown(options=services, value='pendulum', description='Servicio')
depth = widgets.Dropdown(options=depths, value='two_bounce', description='Profundidad')
lane = widgets.Dropdown(options=lanes, value='elbow', description='Zona')

server_x = widgets.FloatSlider(value=530, min=150, max=1250, step=10, description='Bote serv X')
server_y = widgets.FloatSlider(value=TABLE_WIDTH/2, min=0, max=TABLE_WIDTH, step=10, description='Bote serv Y')
opponent_x = widgets.FloatSlider(value=benchmark_direct.DEPTHS['two_bounce']['target_x'], min=TABLE_LENGTH/2, max=TABLE_LENGTH+450, step=10, description='Bote rec X')
opponent_y = widgets.FloatSlider(value=TABLE_WIDTH/2, min=0, max=TABLE_WIDTH, step=10, description='Bote rec Y')
max_height = widgets.FloatSlider(value=235, min=80, max=600, step=5, description='Altura max')
min_height = widgets.FloatSlider(value=120, min=0, max=250, step=5, description='Altura min')
prefer_second = widgets.Checkbox(value=True, description='Preferir 2o bote receptor')
forbid_second = widgets.Checkbox(value=False, description='Evitar 2o bote receptor')

maxiter = widgets.IntSlider(value=60, min=5, max=300, step=5, description='Iteraciones')
popsize = widgets.IntSlider(value=8, min=3, max=30, step=1, description='Población')
dt = widgets.FloatSlider(value=0.005, min=0.0025, max=0.02, step=0.0025, readout_format='.4f', description='dt')
t_max = widgets.FloatSlider(value=3.0, min=1.0, max=5.0, step=0.25, description='t max')
seed = widgets.IntText(value=7, description='Seed')
workers = widgets.IntSlider(value=1, min=1, max=8, step=1, description='Workers')
polish = widgets.Checkbox(value=True, description='Pulir resultado')

run_button = widgets.Button(description='Buscar parámetros', button_style='success')
progress = widgets.IntProgress(value=0, min=0, max=1, description='En espera', bar_style='')
progress_status = widgets.HTML(value='<i>La búsqueda todavía no ha comenzado.</i>')
out = widgets.Output()


In [ ]:
def apply_preset(_=None):
    targets = target_from_benchmark(depth.value, lane.value)
    server_x.value = targets.server_bounce_x
    server_y.value = targets.server_bounce_y
    opponent_x.value = targets.opponent_bounce_x
    opponent_y.value = targets.opponent_bounce_y
    max_height.value = targets.max_height_after_net
    min_height.value = targets.min_height_after_net
    prefer_second.value = targets.prefer_second_opponent_bounce
    forbid_second.value = targets.forbid_second_opponent_bounce

for widget in (depth, lane):
    widget.observe(apply_preset, names='value')
apply_preset()


def build_targets():
    return ServeTargets(
        server_bounce_x=server_x.value,
        server_bounce_y=server_y.value,
        opponent_bounce_x=opponent_x.value,
        opponent_bounce_y=opponent_y.value,
        max_height_after_net=max_height.value,
        min_height_after_net=min_height.value,
        prefer_second_opponent_bounce=prefer_second.value,
        forbid_second_opponent_bounce=forbid_second.value,
    )


def show_result(result):
    print(f"Optimizador: {result.optimizer}")
    print(f"Costo: {result.cost:.4f}; éxito: {result.success}")
    if result.message:
        print(result.message)
    print("\nMétricas")
    print(json.dumps(result.to_json_dict()['metrics'], indent=2, ensure_ascii=False))
    print("\nParámetros")
    print(json.dumps(result.to_json_dict()['parameters'], indent=2, ensure_ascii=False))


def update_progress(update):
    labels = {'global': 'Global', 'polish': 'Pulido', 'complete': 'Listo'}
    progress.max = max(1, update.total)
    progress.value = min(update.current, progress.max)
    progress.description = labels.get(update.phase, update.phase)
    progress.bar_style = 'success' if update.phase == 'complete' else 'info'
    cost = '' if update.best_cost is None else f' · mejor costo: {update.best_cost:.4f}'
    progress_status.value = f'<b>{progress.description}</b>: {update.current}/{update.total}{cost}<br><small>{update.message}</small>'


def on_run(_):
    run_button.disabled = True
    progress.max = maxiter.value
    progress.value = 0
    progress.description = 'Global'
    progress.bar_style = 'info'
    progress_status.value = '<b>Preparando la población…</b>'
    try:
        with out:
            clear_output(wait=True)
            print('Búsqueda iniciada. Observa la barra de progreso debajo del botón.')
            targets = build_targets()
            if mode.value == 'direct':
                config = SearchConfig(
                    targets=targets,
                    weights=SearchWeights(),
                    dt=dt.value,
                    t_max=t_max.value,
                    maxiter=maxiter.value,
                    popsize=popsize.value,
                    polish=polish.value,
                    seed=seed.value,
                    workers=workers.value,
                )
                guess = benchmark_initial_guess(service.value, depth.value, lane.value)
                result = search_direct_parameters(config, initial_guess=guess, progress_callback=update_progress)
            else:
                config = RacketSearchConfig(
                    targets=targets,
                    weights=SearchWeights(),
                    dt=dt.value,
                    t_max=t_max.value,
                    maxiter=maxiter.value,
                    popsize=popsize.value,
                    polish=polish.value,
                    seed=seed.value,
                    workers=workers.value,
                )
                guess = benchmark_racket_initial_guess(service.value, depth.value, lane.value)
                result = search_racket_parameters(config, initial_guess=guess, progress_callback=update_progress)
            show_result(result)
    except Exception as exc:
        progress.bar_style = 'danger'
        progress_status.value = f'<b>Error:</b> {exc}'
        with out:
            print(f'La búsqueda falló: {exc}')
        raise
    finally:
        run_button.disabled = False

run_button.on_click(on_run)


In [ ]:
ui = widgets.VBox([
    widgets.HBox([mode, service, depth, lane]),
    widgets.HTML('<b>Restricciones y objetivos</b>'),
    widgets.HBox([server_x, server_y]),
    widgets.HBox([opponent_x, opponent_y]),
    widgets.HBox([max_height, min_height]),
    widgets.HBox([prefer_second, forbid_second]),
    widgets.HTML('<b>Búsqueda</b>'),
    widgets.HBox([maxiter, popsize]),
    widgets.HBox([dt, t_max]),
    widgets.HBox([seed, workers, polish]),
    run_button,
    progress,
    progress_status,
    out,
])

display(ui)
